<a href="https://colab.research.google.com/github/PankajShukla/Best-Astral-Frame-Selection/blob/main/colab_notebook_to_pdf_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
# Resolving nbconvert issue

# texlive-xetex: Installs the missing xelatex engine your error is complaining about.
# texlive-fonts-recommended: Ensures standard fonts are available so your tables and text render beautifully.
# texlive-plain-generic: Provides basic layout files necessary for nbconvert templates to function without throwing new template errors.
!apt-get update
!apt-get install texlive-xetex texlive-fonts-recommended texlive-plain-generic pandoc

# Install playwright and its Chromium browser
!pip install playwright
!playwright install chromium



Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pandoc is already the newest version (2.9.2.1-3ubuntu2).
pandoc set to manually installed.
tex

In [63]:
# ============================================================
# Google Drive Notebook Collector & PDF Exporter
# ============================================================

import os
import json
import time
import shutil
import subprocess
from pathlib import Path
from datetime import datetime


In [20]:

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------


In [21]:
# Mount Google Drive first:
from google.colab import drive
# drive.flush_and_unmount()

In [ ]:
!google-colab-unmount

/bin/bash: line 1: google-colab-unmount: command not found


In [57]:

drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [58]:



DRIVE_ROOT = "/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/"

# Folder to search.
# Use DRIVE_ROOT to search entire drive.
SEARCH_ROOT = DRIVE_ROOT

# Output folder
OUTPUT_FOLDER = os.path.join(DRIVE_ROOT, "all_notebooks_pdf")

# Status JSON file
STATUS_JSON = os.path.join(
    OUTPUT_FOLDER,
    f"conversion_status_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
)

In [64]:
import os
import shutil
import subprocess
from pathlib import Path
import tempfile

def sanitize_filename(name: str) -> str:
    invalid_chars = '<>:"/\\|?*'
    for c in invalid_chars:
        name = name.replace(c, "_")
    return name


def generate_output_name(notebook_path):
    notebook = Path(notebook_path)

    folder_name = notebook.parent.name
    notebook_name = notebook.stem

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    pdf_name = (
        f"{sanitize_filename(folder_name)}"
        f"_{sanitize_filename(notebook_name)}"
        f"_{timestamp}.pdf"
    )

    return os.path.join(OUTPUT_FOLDER, pdf_name)


def convert_notebook_to_pdf(notebook_path, output_pdf_path):
    """
    Convert notebook to PDF.
    Primary approach: LaTeX (--to pdf)
    Fallback approach: Headless Browser (--to webpdf)
    """
    notebook_path = os.path.abspath(notebook_path)
    output_pdf_path = os.path.abspath(output_pdf_path)
    pdf_filename = Path(output_pdf_path).name

    # Use a secure, unique temporary directory to prevent conflicts
    with tempfile.TemporaryDirectory(prefix="nb_pdf_") as temp_dir:

        # --- APPROACH 1: LaTeX (--to pdf) ---
        print("Attempting primary conversion method: LaTeX...")
        latex_command = [
            "jupyter", "nbconvert",
            "--to", "pdf",
            notebook_path,
            "--output", pdf_filename.replace(".pdf", ""),
            "--output-dir", temp_dir,
            "--no-prompt",
            "--allow-errors",
            "--ExecutePreprocessor.timeout=600",
        ]

        result = subprocess.run(latex_command, capture_output=True, text=True)

        # If LaTeX succeeds, verify and move the file
        if result.returncode == 0:
            generated_pdf = os.path.join(temp_dir, pdf_filename)
            if os.path.exists(generated_pdf):
                os.makedirs(os.path.dirname(output_pdf_path), exist_ok=True)
                shutil.move(generated_pdf, output_pdf_path)
                print("Successfully converted using LaTeX.")
                return output_pdf_path

        # --- APPROACH 2: Fallback to webpdf ---
        print("LaTeX conversion failed or file not found. Falling back to webpdf...")

        webpdf_command = [
            "jupyter", "nbconvert",
            "--to", "webpdf",
            notebook_path,
            "--output", pdf_filename.replace(".pdf", ""),
            "--output-dir", temp_dir,
            "--no-prompt",
            "--allow-errors",
            "--ExecutePreprocessor.timeout=600",
        ]

        fallback_result = subprocess.run(webpdf_command, capture_output=True, text=True)

        if fallback_result.returncode != 0:
            # Both methods failed, construct a detailed error log
            error_message = (
                f"Both conversion methods failed.\n\n"
                f"--- LATEX ERROR ---\nCode: {result.returncode}\nSTDERR: {result.stderr}\n\n"
                f"--- WEBPDF ERROR ---\nCode: {fallback_result.returncode}\nSTDERR: {fallback_result.stderr}"
            )
            raise RuntimeError(error_message)

        # Handle webpdf output mapping
        generated_pdf = os.path.join(temp_dir, pdf_filename)
        if not os.path.exists(generated_pdf):
            candidates = [os.path.join(temp_dir, f) for f in os.listdir(temp_dir) if f.endswith(".pdf")]
            if not candidates:
                raise RuntimeError("webpdf command reported success, but no PDF was generated.")
            generated_pdf = candidates[0]

        os.makedirs(os.path.dirname(output_pdf_path), exist_ok=True)
        shutil.move(generated_pdf, output_pdf_path)
        print("Successfully converted using webpdf fallback.")

        return output_pdf_path

In [60]:

# ------------------------------------------------------------
# FIND NOTEBOOKS
# ------------------------------------------------------------

notebooks = []

for root, dirs, files in os.walk(SEARCH_ROOT):
    for file in files:
        if file.endswith(".ipynb"):
            notebooks.append(os.path.join(root, file))

print(f"Found {len(notebooks)} notebooks")


Found 98 notebooks


In [65]:
notebooks[:10]

['/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/M6_NB_MiniProject_4_AirQuality_Forecast_ARIMA.ipynb',
 '/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/module6_exam_2236265.ipynb',
 '/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/CDS Module 1/M1_AST_02_Bayes_Rule_A_.ipynb',
 '/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/CDS Module 1/Pankaj_M1_MP1_NB_Resume_Classification_Using_Naive_Bayes.ipynb',
 '/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/CDS Module 1/M1_AST_03_Bayesian_Inference_A.ipynb',
 '/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/CDS Module 1/M1_AST_04_Text_Analysis_A.ipynb',
 '/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/CDS Module 1/M1_AST_05_Probability_distributions_C.ipynb',
 '/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/CDS Module 1/M1_AST_06_Statistical_Testing_C.ipynb',
 '/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/CDS Module 1/M1_AST_01_Probability_Basics_A (1).ipyn

In [66]:
STATUS_JSON = '/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/all_notebooks_pdf/conversion_status_20260615_070546.json'
print(STATUS_JSON)

/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/all_notebooks_pdf/conversion_status_20260615_070546.json


In [67]:
import os
import json
import time
from datetime import datetime

# Initialize status_log (if not already defined globally in a previous cell)
# It's better to ensure it's initialized before the loop if it's meant to accumulate across runs.
# If status_log is defined in an earlier cell, this part should be adjusted.

# Load existing status log if it exists
if os.path.exists(STATUS_JSON):
    with open(STATUS_JSON, "r") as f:
        status_log = json.load(f)
    print(f"Loaded {len(status_log)} entries from existing status log.")
else:
    status_log = []

# Create a set of already successfully processed notebook paths for quick lookup
successful_notebooks = {entry['notebook_path'] for entry in status_log if entry['status'] == 'success'}
print("status : ", status_log)
print(successful_notebooks)
print(len(successful_notebooks))

Loaded 113 entries from existing status log.
status :  [{'notebook_path': '/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/M6_NB_MiniProject_4_AirQuality_Forecast_ARIMA.ipynb', 'status': 'fail', 'timestamp': '2026-06-15T07:22:12.664014', 'reason': '[NbConvertApp] Converting notebook /content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/M6_NB_MiniProject_4_AirQuality_Forecast_ARIMA.ipynb to pdf\n[NbConvertApp] Writing 35259 bytes to notebook.tex\n[NbConvertApp] Building PDF\nTraceback (most recent call last):\n  File "/usr/local/bin/jupyter-nbconvert", line 10, in <module>\n    sys.exit(main())\n             ^^^^^^\n  File "/usr/local/lib/python3.12/dist-packages/jupyter_core/application.py", line 284, in launch_instance\n    super().launch_instance(argv=argv, **kwargs)\n  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance\n    app.start()\n  File "/usr/local/lib/python3.12/dist-packages/nbconvert/nbconvertapp.py", li

In [68]:

# ------------------------------------------------------------
# PROCESS NOTEBOOKS
# ------------------------------------------------------------

for idx, notebook_path in enumerate(notebooks, start=1):

    print(f"[{idx}/{len(notebooks)}] Processing:")
    print(notebook_path)

    # Check if this notebook has already been successfully processed
    if notebook_path in successful_notebooks:
        print("  ✓ Already successfully converted. Skipping.")
        continue # Skip to the next notebook

    start_time = time.time()

    entry = {
        "notebook_path": notebook_path,
        "status": None,
        "timestamp": datetime.now().isoformat(),
        "reason": None,
        "time_spent_seconds": None,
        "pdf_path": None,
    }

    try:

        output_pdf = generate_output_name(notebook_path)

        convert_notebook_to_pdf(
            notebook_path,
            output_pdf
        )

        entry["status"] = "success"
        entry["pdf_path"] = output_pdf

        print("  ✓ Success")

    except Exception as e:

        entry["status"] = "fail"
        entry["reason"] = str(e)

        print("  ✗ Failed")
        # print(f"    {e}")

    finally:

        entry["time_spent_seconds"] = round(
            time.time() - start_time,
            2
        )

        status_log.append(entry)

        with open(STATUS_JSON, "w") as f:
            json.dump(
                status_log,
                f,
                indent=2,
                ensure_ascii=False
            )


[1/98] Processing:
/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/M6_NB_MiniProject_4_AirQuality_Forecast_ARIMA.ipynb
  ✓ Already successfully converted. Skipping.
[2/98] Processing:
/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/module6_exam_2236265.ipynb
  ✓ Already successfully converted. Skipping.
[3/98] Processing:
/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/CDS Module 1/M1_AST_02_Bayes_Rule_A_.ipynb
Attempting primary conversion method: LaTeX...
LaTeX conversion failed or file not found. Falling back to webpdf...
  ✗ Failed
[4/98] Processing:
/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/CDS Module 1/Pankaj_M1_MP1_NB_Resume_Classification_Using_Naive_Bayes.ipynb
  ✓ Already successfully converted. Skipping.
[5/98] Processing:
/content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/CDS Module 1/M1_AST_03_Bayesian_Inference_A.ipynb
  ✓ Already successfully converted. Skipping.
[6/98] Processing:
/content/drive/MyDrive/Data Scientist/IISc

In [69]:

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

success_count = sum(
    1 for x in status_log
    if x["status"] == "success"
)

fail_count = sum(
    1 for x in status_log
    if x["status"] == "fail"
)


In [70]:

print("\n==============================")
print("COMPLETED")
print("==============================")
print(f"Total notebooks : {len(status_log)}")
print(f"Success         : {success_count}")
print(f"Failed          : {fail_count}")
print(f"PDF folder      : {OUTPUT_FOLDER}")
print(f"Status JSON     : {STATUS_JSON}")


COMPLETED
Total notebooks : 154
Success         : 61
Failed          : 89
PDF folder      : /content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/all_notebooks_pdf
Status JSON     : /content/drive/MyDrive/Data Scientist/IISc CDS Notebooks/all_notebooks_pdf/conversion_status_20260615_070546.json
